In [23]:
# 경로 및 설정
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import yaml
from IPython.display import clear_output, display
from tqdm.auto import tqdm


REPO_ROOT = Path("/home/stu03/workspace/model")

CONFIG_PATH = (
    REPO_ROOT
    / "configs/experiments/hemorrhage/H-N30.yaml"
)

DATA_ROOT = Path(
    "/home/stu03/workspace/datasets/hemorrhage/v1/"
    "model_ready/nnunet"
)

RESULTS_ROOT = REPO_ROOT / "runs/nnunet"

config = yaml.safe_load(
    CONFIG_PATH.read_text(encoding="utf-8")
)

experiment = config["experiment"]
plans = config["plans"]

dataset_name = (
    f"Dataset{int(experiment['dataset_id']):03d}_"
    f"{experiment.get('dataset_name', 'ICHBinary')}"
)

trainer_folder = (
    "nnUNetTrainerICHBaseline"
    f"__{plans['output_identifier']}"
    f"__{experiment['configuration']}"
)

OUTPUT_FOLDER = (
    RESULTS_ROOT
    / dataset_name
    / trainer_folder
    / f"fold_{experiment['fold']}"
)

PROGRESS_PATH = OUTPUT_FOLDER / "progress.json"
METRICS_PATH = OUTPUT_FOLDER / "custom_metrics.json"
LOG_PATH = RESULTS_ROOT / "H-N30-launch.log"
PID_PATH = RESULTS_ROOT / "H-N30.pid"

PLAN_PATH = (
    DATA_ROOT
    / "nnUNet_preprocessed"
    / dataset_name
    / f"{plans['output_identifier']}.json"
)

print("Config:", CONFIG_PATH)
print("Plan:", PLAN_PATH)
print("Output:", OUTPUT_FOLDER)
print("Progress:", PROGRESS_PATH)
print("Log:", LOG_PATH)

Config: /home/stu03/workspace/model/configs/experiments/hemorrhage/H-N30.yaml
Plan: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset001_ICHBinary/ICHPlansDefaultAttentionV1.json
Output: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0
Progress: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0/progress.json
Log: /home/stu03/workspace/model/runs/nnunet/H-N30-launch.log


In [24]:
from copy import deepcopy
from pathlib import Path
import tempfile
import yaml


probe_config = deepcopy(config)

probe_config["experiment"]["name"] = (
    "H-N30-vram-probe"
)

probe_config["trainer"]["epochs"] = 1
probe_config["trainer"]["iterations_per_epoch"] = 1
probe_config["trainer"]["validation_iterations_per_epoch"] = 1
probe_config["trainer"]["save_every"] = 1

probe_config["pruning"]["enabled"] = False

probe_directory = Path(
    tempfile.mkdtemp(prefix="brainon-hn30-vram-")
)

probe_config_path = (
    probe_directory / "H-N30-vram-probe.yaml"
)

probe_results_root = (
    probe_directory / "results"
)

probe_config_path.write_text(
    yaml.safe_dump(
        probe_config,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("Probe config:", probe_config_path)
print("Probe results:", probe_results_root)

Probe config: /tmp/brainon-hn30-vram-6k3rsx5a/H-N30-vram-probe.yaml
Probe results: /tmp/brainon-hn30-vram-6k3rsx5a/results


In [25]:
import os
import subprocess
import sys


probe_code = r"""
import json
import torch

from brainon_ai.nnunet.runner import run_from_config


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

result = {}

try:
    output = run_from_config(
        config_path=__CONFIG_PATH__,
        data_root=__DATA_ROOT__,
        results_root=__RESULTS_ROOT__,
        run_validation=False,
    )

    result["status"] = "completed"
    result["output"] = str(output)

except torch.cuda.OutOfMemoryError as exc:
    result["status"] = "out_of_memory"
    result["error"] = str(exc)

except Exception as exc:
    result["status"] = "failed"
    result["error"] = repr(exc)

finally:
    result["peak_allocated_gb"] = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
    result["peak_reserved_gb"] = (
        torch.cuda.max_memory_reserved()
        / 1024**3
    )

    free_bytes, total_bytes = (
        torch.cuda.mem_get_info()
    )

    result["free_gb_after"] = (
        free_bytes / 1024**3
    )
    result["total_gb"] = (
        total_bytes / 1024**3
    )

    print(
        "VRAM_PROBE_RESULT="
        + json.dumps(result)
    )
"""

probe_code = (
    probe_code
    .replace(
        "__CONFIG_PATH__",
        repr(str(probe_config_path)),
    )
    .replace(
        "__DATA_ROOT__",
        repr(str(DATA_ROOT)),
    )
    .replace(
        "__RESULTS_ROOT__",
        repr(str(probe_results_root)),
    )
)

probe_environment = os.environ.copy()

probe_environment["nnUNet_n_proc_DA"] = "0"
probe_environment["nnUNet_def_n_proc"] = "1"

probe_environment["PYTHONPATH"] = (
    os.pathsep.join(
        [
            str(REPO_ROOT / "src"),
            probe_environment.get(
                "PYTHONPATH",
                "",
            ),
        ]
    ).rstrip(os.pathsep)
)

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        probe_code,
    ],
    cwd=REPO_ROOT,
    env=probe_environment,
    text=True,
    capture_output=True,
)

print(probe.stdout)

if probe.stderr:
    print("STDERR:")
    print(probe.stderr[-5000:])

print("Return code:", probe.returncode)

Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##############################################################

In [26]:
# 실행 전 검사
required_paths = [
    CONFIG_PATH,
    DATA_ROOT / "nnUNet_raw",
    DATA_ROOT / "nnUNet_preprocessed",
    PLAN_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "필수 경로가 없습니다:\n"
        + "\n".join(missing)
    )

assert config["network"]["name"] == "default_attention"
assert plans["base_identifier"] == "nnUNetPlans"
assert plans["output_identifier"] == "ICHPlansDefaultAttentionV1"

print("H-N30 실행 준비 완료")

H-N30 실행 준비 완료


In [27]:
# dry run
environment = os.environ.copy()

environment["nnUNet_n_proc_DA"] = "0"
environment["nnUNet_def_n_proc"] = "1"

environment["PYTHONPATH"] = os.pathsep.join(
    [
        str(REPO_ROOT / "src"),
        environment.get("PYTHONPATH", ""),
    ]
).rstrip(os.pathsep)

command = [
    sys.executable,
    "-m",
    "brainon_ai.nnunet.runner",
    "--config",
    str(CONFIG_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--results-root",
    str(RESULTS_ROOT),
    "--dry-run",
]

result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        f"dry-run 실패: returncode={result.returncode}"
    )

{
  "config": "/home/stu03/workspace/model/configs/experiments/hemorrhage/H-N30.yaml",
  "data_paths": {
    "nnUNet_raw": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_raw",
    "nnUNet_preprocessed": "/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed",
    "nnUNet_results": "/home/stu03/workspace/model/runs/nnunet"
  },
  "output_folder": "/home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0",
  "experiment": {
    "name": "H-N30-nnunet-default-attention",
    "dataset_id": 1,
    "dataset_name": "ICHBinary",
    "configuration": "3d_fullres",
    "fold": 0,
    "device": "auto",
    "continue_training": false,
    "validation_checkpoint": "best_composite"
  },
  "plans": {
    "base_identifier": "nnUNetPlans",
    "output_identifier": "ICHPlansDefaultAttentionV1",
    "patch_size": [
      20,
      320,
      256
    ],
    "batch_size": 1
  },

In [28]:
# 기존 실행 충돌 및 resume 확인
config = yaml.safe_load(
    CONFIG_PATH.read_text(encoding="utf-8")
)

continue_training = bool(
    config["experiment"].get(
        "continue_training",
        False,
    )
)

print("continue_training:", continue_training)
print("output folder:", OUTPUT_FOLDER)

if OUTPUT_FOLDER.exists():
    existing_files = sorted(
        path.name
        for path in OUTPUT_FOLDER.iterdir()
    )

    print("기존 output 폴더가 있습니다.")
    print(existing_files)

    latest_checkpoint = (
        OUTPUT_FOLDER / "checkpoint_latest.pth"
    )

    final_checkpoint = (
        OUTPUT_FOLDER / "checkpoint_final.pth"
    )

    print(
        "checkpoint_latest:",
        latest_checkpoint.exists(),
    )
    print(
        "checkpoint_final:",
        final_checkpoint.exists(),
    )

    if continue_training:
        if not latest_checkpoint.exists():
            raise RuntimeError(
                "continue_training=true이지만 "
                "checkpoint_latest.pth가 없습니다."
            )

        print(
            "재개 가능한 checkpoint를 확인했습니다."
        )

    else:
        raise RuntimeError(
            "기존 H-N30 결과가 있습니다. "
            "새 학습이면 결과 폴더를 삭제하고, "
            "재개하려면 H-N30.yaml에서 "
            "continue_training: true로 바꾸세요."
        )

else:
    if continue_training:
        raise RuntimeError(
            "continue_training=true이지만 "
            "기존 H-N30 결과 폴더가 없습니다."
        )

    print(
        "기존 결과와 충돌하지 않습니다. "
        "새 학습을 시작할 수 있습니다."
    )

continue_training: False
output folder: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0
기존 결과와 충돌하지 않습니다. 새 학습을 시작할 수 있습니다.


In [29]:
# H-N30 학습
RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

train_command = [
    sys.executable,
    "-m",
    "brainon_ai.nnunet.runner",
    "--config",
    str(CONFIG_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--results-root",
    str(RESULTS_ROOT),
]

log_handle = LOG_PATH.open(
    "a",
    encoding="utf-8",
    buffering=1,
)

log_handle.write(
    "\n\n===== H-N30 launch =====\n"
)
log_handle.write(
    " ".join(train_command) + "\n"
)
log_handle.flush()

process = subprocess.Popen(
    train_command,
    cwd=REPO_ROOT,
    env=environment,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

PID_PATH.write_text(
    str(process.pid),
    encoding="utf-8",
)

print("H-N30 학습 시작")
print("PID:", process.pid)
print("Log:", LOG_PATH)
print("Progress:", PROGRESS_PATH)

H-N30 학습 시작
PID: 821190
Log: /home/stu03/workspace/model/runs/nnunet/H-N30-launch.log
Progress: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0/progress.json


In [30]:
# 프로세스 즉시 확인
time.sleep(3)

return_code = process.poll()

if return_code is None:
    print("학습 프로세스 실행 중")
else:
    print(
        f"학습 프로세스 조기 종료: {return_code}"
    )

    if LOG_PATH.exists():
        lines = LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        print("\n".join(lines[-80:]))

학습 프로세스 실행 중


In [31]:
# 실시간 진행률 표시
total_epochs = int(
    config["trainer"]["epochs"]
)

terminal_statuses = {
    "completed",
    "early_stopped",
    "failed",
}

bar = tqdm(
    total=total_epochs,
    desc="H-N30 epochs",
    unit="epoch",
)

last_epoch = 0

while True:
    state = {}

    if PROGRESS_PATH.exists():
        try:
            state = json.loads(
                PROGRESS_PATH.read_text(
                    encoding="utf-8"
                )
            )
        except json.JSONDecodeError:
            time.sleep(1)
            continue

        current_epoch = int(
            state.get("current_epoch", 0)
        )

        if current_epoch > last_epoch:
            bar.update(
                current_epoch - last_epoch
            )
            last_epoch = current_epoch

        bar.set_postfix(
            status=state.get("status"),
            dice=state.get("pseudo_dice"),
            recall=state.get("pseudo_recall"),
            composite=state.get("composite"),
            best_dice=state.get("best_dice"),
            best_composite=state.get(
                "best_composite"
            ),
            bad_checks=state.get(
                "pruning_bad_checks"
            ),
            patience=state.get(
                "pruning_patience"
            ),
        )

    # 현재 커널에서 시작한 프로세스가 있으면 상태 확인
    process_return_code = None

    if "process" in globals():
        process_return_code = process.poll()

        if process_return_code is not None:
            break

    status = state.get("status")

    if status in terminal_statuses:
        break

    clear_output(wait=True)

    display(
        {
            "status": status or "starting",
            "epoch": (
                f"{state.get('current_epoch', 0)}"
                f"/{state.get('total_epochs', total_epochs)}"
            ),
            "progress_percent": state.get(
                "progress_percent", 0
            ),
            "eta_seconds": state.get(
                "eta_seconds"
            ),
            "pseudo_dice": state.get(
                "pseudo_dice"
            ),
            "pseudo_recall": state.get(
                "pseudo_recall"
            ),
            "composite": state.get(
                "composite"
            ),
            "best_dice": state.get(
                "best_dice"
            ),
            "best_recall": state.get(
                "best_recall"
            ),
            "best_composite": state.get(
                "best_composite"
            ),
            "early_stopping": (
                f"{state.get('pruning_bad_checks', 0)}"
                f"/{state.get('pruning_patience', 30)}"
            ),
            "checkpoint_latest": (
                OUTPUT_FOLDER
                / "checkpoint_latest.pth"
            ).exists(),
        }
    )

    time.sleep(5)

bar.close()

if "process" in globals():
    print(
        "process return code:",
        process.poll(),
    )

if PROGRESS_PATH.exists():
    final_state = json.loads(
        PROGRESS_PATH.read_text(
            encoding="utf-8"
        )
    )

    display(final_state)

{'status': 'running',
 'epoch': '249/300',
 'progress_percent': 83.0,
 'eta_seconds': 1008.2468970970936,
 'pseudo_dice': 0.8153986930847168,
 'pseudo_recall': 0.8210082054138184,
 'composite': 0.8182034492492676,
 'best_dice': 0.8415229916572571,
 'best_recall': 0.8962280750274658,
 'best_composite': 0.8554658889770508,
 'early_stopping': '29/30',
 'checkpoint_latest': True}

H-N30 epochs:  83%|████████▎ | 250/300 [1:24:27<16:53, 20.27s/epoch, bad_checks=30, best_composite=0.855, best_dice=0.842, composite=None, dice=None, patience=30, recall=None, status=early_stopped]

process return code: None


{'status': 'early_stopped',
 'message': None,
 'updated_at': '2026-09-01T09:47:44.096269+00:00',
 'current_epoch': 250,
 'total_epochs': 300,
 'progress_percent': 83.33,
 'session_elapsed_seconds': 5073.0,
 'last_epoch_seconds': 15.708570957183838,
 'mean_epoch_seconds': 19.753303097724913,
 'eta_seconds': 987.6651548862457,
 'pseudo_dice': None,
 'pseudo_recall': None,
 'composite': None,
 'best_dice': 0.8415229916572571,
 'best_recall': 0.8962280750274658,
 'best_composite': 0.8554658889770508,
 'pruning_bad_checks': 30,
 'pruning_patience': 30,
 'output_folder': '/home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansDefaultAttentionV1__3d_fullres/fold_0',
 'batch_size': 1,
 'patch_size': [20, 320, 256]}

In [ ]:
# 로그 마지막 부분 확인
if not LOG_PATH.exists():
    print("로그 파일이 아직 없습니다.")
else:
    lines = LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    print("\n".join(lines[-100:]))

In [ ]:
# 학습 후 결과 확인
if METRICS_PATH.exists():
    metrics = json.loads(
        METRICS_PATH.read_text(
            encoding="utf-8"
        )
    )

    print("Patch-based online metrics")
    display(metrics)
else:
    print(
        "custom_metrics.json이 아직 없습니다."
    )

validation_summary = (
    OUTPUT_FOLDER
    / "validation"
    / "summary.json"
)

if validation_summary.exists():
    summary = json.loads(
        validation_summary.read_text(
            encoding="utf-8"
        )
    )

    print("Full-volume validation")
    display(summary["foreground_mean"])
else:
    print(
        "Full-volume validation summary가 "
        "아직 없습니다."
    )